# agentic-civic-resolution-app 
Agent 3: Department Routing Agent

**Role:** Classification + Severity → assigned department, officer tier, escalation flag, action plan

 This is the terminal agent in the pipeline. Its output drives:
 Ticket assignment in the ops system
 Escalation notifications (Slack / email via Day 5)
 Dashboard queue population

**COMMAND** 

 %pip install databricks-langchain mlflow pydantic --quiet

 dbutils.library.restartPython()

In [0]:
%pip install databricks-langchain mlflow pydantic --quiet

#dbutils.library.restartPython()

## 0. Config & Imports

In [0]:
import json, re, mlflow, requests
from pydantic import BaseModel, Field
from databricks_langchain import ChatDatabricks

In [0]:
def ensure_mlflow_experiment(experiment_path: str) -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    parts = experiment_path.strip("/").split("/")
    for depth in range(1, len(parts)):
        folder = "/" + "/".join(parts[:depth])
        resp   = requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=headers, json={"path": folder})
        body   = resp.json()
        if resp.status_code == 200:
            print(f"  ✓ folder: {folder}")
        elif body.get("error_code") == "RESOURCE_ALREADY_EXISTS":
            print(f"  ✓ exists: {folder}")
        else:
            raise RuntimeError(f"mkdirs failed for {folder}: {body}")

    exp = mlflow.set_experiment(experiment_path)
    return exp.experiment_id

EXPERIMENT_ID = ensure_mlflow_experiment("/civicops/agents/department_router")
print(f"✓ MLflow experiment ready (id={EXPERIMENT_ID})")


In [0]:
def discover_llm_endpoint() -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}"}
    resp    = requests.get(f"{host}/api/2.0/serving-endpoints", headers=headers)
    ready   = [e for e in resp.json().get("endpoints", []) if e.get("state", {}).get("ready") == "READY"]
    if not ready:
        raise RuntimeError("No READY serving endpoints found. Go to Serving → create one first.")
    for kw in ["llama", "dbrx", "mixtral", "mpt"]:
        for e in ready:
            if kw in e["name"].lower():
                print(f"✓ Auto-selected: {e['name']}")
                return e["name"]
    return ready[0]["name"]

LLM_ENDPOINT = discover_llm_endpoint()

## 1. Department Registry

 Centralised lookup — mirrors what would live in Lakebase (Day 3).

In [0]:
DEPARTMENT_REGISTRY = {
    "Sanitation":         {
        "dept_code": "SWMD",
        "dept_name": "Solid Waste Management Department",
        "contact":   "swmd@civicops.city",
        "escalation_dept": "Municipal Commissioner Office",
    },
    "Infrastructure": {
        "dept_code": "PWD",
        "dept_name": "Public Works Department",
        "contact":   "pwd@civicops.city",
        "escalation_dept": "Chief Engineer Office",
    },
    "Noise": {
        "dept_code": "PCB",
        "dept_name": "Pollution Control Board",
        "contact":   "pcb@civicops.city",
        "escalation_dept": "District Magistrate Office",
    },
    "Water & Sewage": {
        "dept_code": "BWSSB",
        "dept_name": "Water Supply & Sewerage Board",
        "contact":   "bwssb@civicops.city",
        "escalation_dept": "Water Resources Ministry",
    },
    "Electricity": {
        "dept_code": "BESCOM",
        "dept_name": "Electricity Supply Company",
        "contact":   "bescom@civicops.city",
        "escalation_dept": "Energy Department",
    },
    "Road & Traffic": {
        "dept_code": "BBMP-TE",
        "dept_name": "Traffic Engineering Division",
        "contact":   "bbmp-te@civicops.city",
        "escalation_dept": "Commissioner of Police (Traffic)",
    },
    "Public Safety": {
        "dept_code": "POLICE",
        "dept_name": "City Police Department",
        "contact":   "police@civicops.city",
        "escalation_dept": "Deputy Commissioner of Police",
    },
    "Parks & Environment": {
        "dept_code": "BBMP-HP",
        "dept_name": "Horticulture & Parks Division",
        "contact":   "bbmp-hp@civicops.city",
        "escalation_dept": "Forest & Environment Department",
    },
}

# Officer tier by severity score
OFFICER_TIER = {
    5: "Commissioner / Director Level",
    4: "Deputy Commissioner Level",
    3: "Assistant Commissioner Level",
    2: "Field Supervisor Level",
    1: "Field Officer Level",
}

## 2. Output Schema

In [0]:
class RoutingOutput(BaseModel):
    dept_code:         str  = Field(description="Short department code e.g. SWMD, PWD")
    dept_name:         str  = Field(description="Full department name")
    dept_contact:      str  = Field(description="Department contact email")
    officer_tier:      str  = Field(description="Required officer seniority level")
    escalate:          bool = Field(description="True if complaint needs escalation beyond primary dept")
    escalation_dept:   str | None = Field(description="Escalation department if escalate=True")
    action_plan:       list[str]  = Field(description="Ordered list of 3-5 concrete action steps")
    field_visit_needed: bool = Field(description="True if a physical site inspection is needed")
    notify_citizen:    bool = Field(description="True if citizen should get an immediate SMS/email")
    reasoning:         str  = Field(description="One sentence justifying the routing decision")

## 3. System Prompt

In [0]:

SYSTEM_PROMPT = """
You are the Department Routing Agent for CivicOps AI.

Given a complaint's category, subcategory, severity score (1-5), and raw text, you must:
1. Confirm the correct department from the registry provided.
2. Determine if escalation is needed (severity >= 4 OR health/infrastructure risk = true).
3. Generate a concrete, ordered action plan (3-5 steps).
4. Decide if field visit and citizen notification are required.

Escalation rule: severity_score >= 4 OR health_risk = true => escalate = true.

Respond ONLY with valid JSON - no markdown fences, no extra text:
{
  "dept_code": "<string>",
  "dept_name": "<string>",
  "dept_contact": "<string>",
  "officer_tier": "<string>",
  "escalate": <true|false>,
  "escalation_dept": "<string or null>",
  "action_plan": ["step 1", "step 2", "step 3"],
  "field_visit_needed": <true|false>,
  "notify_citizen": <true|false>,
  "reasoning": "<one sentence>"
}
""".strip()

## 4. Agent Class

In [0]:
class DepartmentRouterAgent:
    """Routes a classified + scored complaint to the correct department."""

    def __init__(self, endpoint: str = LLM_ENDPOINT):
        self.llm      = ChatDatabricks(endpoint=endpoint, temperature=0.0)
        self.endpoint = endpoint

    def route(
        self,
        complaint_text:    str,
        category:          str,
        subcategory:       str,
        severity_score:    int,
        health_risk:       bool = False,
        infrastructure_risk: bool = False,
    ) -> RoutingOutput:
        dept_info  = DEPARTMENT_REGISTRY.get(category, DEPARTMENT_REGISTRY["Infrastructure"])
        tier       = OFFICER_TIER.get(severity_score, OFFICER_TIER[1])
        must_escalate = severity_score >= 4 or health_risk

        context = f"""
Complaint Text: {complaint_text.strip()}
Category: {category}
Subcategory: {subcategory}
Severity Score: {severity_score} / 5
Health Risk: {health_risk}
Infrastructure Risk: {infrastructure_risk}

Department Registry Match:
  dept_code:       {dept_info['dept_code']}
  dept_name:       {dept_info['dept_name']}
  dept_contact:    {dept_info['contact']}
  escalation_dept: {dept_info['escalation_dept']}
  officer_tier:    {tier}
  must_escalate:   {must_escalate}
""".strip()

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": context},
        ]

        with mlflow.start_run(run_name="route_complaint", nested=True):
            mlflow.log_params({
                "endpoint":       self.endpoint,
                "category":       category,
                "severity_score": severity_score,
            })
            response = self.llm.invoke(messages)
            result   = self._parse(response.content)

            # Hard-enforce escalation rule
            if must_escalate:
                result.escalate        = True
                result.escalation_dept = result.escalation_dept or dept_info["escalation_dept"]

            mlflow.log_metric("escalate", int(result.escalate))

        return result

    @staticmethod
    def _parse(raw: str) -> RoutingOutput:
        raw = raw.strip()
        try:
            return RoutingOutput(**json.loads(raw))
        except Exception:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                return RoutingOutput(**json.loads(m.group()))
            raise ValueError(f"Unparseable routing output:\n{raw}")


router_agent = DepartmentRouterAgent()
print("✓ Department routing agent ready.")

## 5. Smoke Test

In [0]:
test_cases = [
    dict(complaint_text="Garbage overflow near Whitefield",
         category="Sanitation", subcategory="Garbage Overflow",
         severity_score=3, health_risk=False, infrastructure_risk=False),
    dict(complaint_text="Exposed live wires near the school playground",
         category="Electricity", subcategory="Exposed Wires",
         severity_score=5, health_risk=True, infrastructure_risk=True),
    dict(complaint_text="Water pipeline burst on Main Street — flooded since 2 days",
         category="Water & Sewage", subcategory="Water Leakage",
         severity_score=4, health_risk=True, infrastructure_risk=False),
]

for tc in test_cases:
    r = router_agent.route(**tc)
    print(f"\nComplaint : {tc['complaint_text']}")
    print(f"Department: [{r.dept_code}] {r.dept_name}")
    print(f"Officer   : {r.officer_tier}")
    print(f"Escalate  : {r.escalate} → {r.escalation_dept}")
    print(f"Action Plan:")
    for i, step in enumerate(r.action_plan, 1):
        print(f"  {i}. {step}")
    print(f"Field Visit: {r.field_visit_needed} | Notify Citizen: {r.notify_citizen}")

## 6. Register in MLflow Model Registry

In [0]:

import pandas as pd
from mlflow.models.signature import infer_signature

class _RouterWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.agent = DepartmentRouterAgent()

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in model_input.iterrows():
            r = self.agent.route(
                complaint_text=str(row["complaint_text"]),
                category=str(row["category"]),
                subcategory=str(row["subcategory"]),
                severity_score=int(row["severity_score"]),
                health_risk=bool(row.get("health_risk", False)),
                infrastructure_risk=bool(row.get("infrastructure_risk", False)),
            )
            results.append(r.model_dump())
        return pd.DataFrame(results)

_input_example  = pd.DataFrame({"complaint_text": ["Garbage overflow"], "category": ["Sanitation"], "subcategory": ["Garbage Overflow"], "severity_score": [3], "health_risk": [False], "infrastructure_risk": [False]})
_output_example = pd.DataFrame([RoutingOutput(
    dept_code="SWMD", dept_name="Solid Waste Management", dept_contact="swmd@civicops.city",
    officer_tier="Field Supervisor Level", escalate=False, escalation_dept=None,
    action_plan=["Inspect site", "Deploy crew", "Confirm clearance"],
    field_visit_needed=True, notify_citizen=True, reasoning="Example."
).model_dump()])
_signature = infer_signature(_input_example, _output_example)

spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.agents")
with mlflow.start_run(run_name="register_router"):
    mlflow.pyfunc.log_model(
        name="department_router",
        python_model=_RouterWrapper(),
        pip_requirements=["databricks-langchain", "pydantic"],
        input_example=_input_example,
        signature=_signature,
        registered_model_name="civicops.agents.department_router",
    )

print("✓ Router agent registered in MLflow Model Registry.")
